# OmniGraph on Kaggle — the second pretraining objective

Self-contained. It clones the repo, installs what Kaggle is missing, runs the
masked-autoencoder (MAE) matrix on a T4, and saves the results for download.

## Before you run

In the panel on the right:

| setting | value |
|---|---|
| **Accelerator** | `GPU T4 x2` or `GPU P100` |
| **Internet** | **ON** — required, or the clone and the dataset downloads both fail |
| **Persistence** | leave off; results are saved as notebook output |

Then **Run All**, or **Save Version → Save & Run All** to let it run in the
background with the tab closed. The second is strongly recommended: the full
matrix takes hours and a browser disconnect will not stop it.

## Why this run exists

The first study (360 runs, on a local RTX 2050) found that a cross-domain
pretrained encoder beats neither a random-init encoder nor a single-source
expert. That result used one self-supervised objective, **DGI**, and DGI
solved its pretext task at step 21 of 300 — after which it supplies no
gradient. So the negative result could mean either:

- self-supervised pretraining does not transfer across these domains, **or**
- DGI specifically is too easy here

This run settles it. **MAE** hides half of each node's features and
reconstructs them from neighbours, which never fully saturates — measured on
Cora, MAE was still improving at step 276 where DGI had finished at step 21.

Everything else is identical: same features, same splits, same frozen probe,
same metrics, same seeds. Only the pretraining objective changes.

## One rule about comparing numbers

Kaggle's T4 is not your local GPU, and its torch version differs. Results are
bit-reproducible *within* a machine, not across machines — so **compare Kaggle
numbers only to Kaggle numbers**. That is why this run re-does arms B and C
here rather than reusing the local ones: every number in the comparison comes
off the same card.

## 1. Get the code

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/kibda/OmniGraph.git"
ROOT = "/kaggle/working/OmniGraph"

if os.path.isdir(ROOT):
    print("repo already present, pulling latest")
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)

os.chdir(ROOT)
sys.path.insert(0, ROOT)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"\nrepo at commit {commit}")

In [ ]:
# Kaggle ships torch with CUDA already -- never reinstall it, that would
# replace a working CUDA build and cost ten minutes. Only the graph library
# is missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch_geometric==2.8.0.post1"], check=True)

from src import env
report = env.print_report()
env.assert_environment_ok(report)

## 2. Choose the scope

The full matrix is 360 runs and takes roughly 7 hours. Kaggle allows 12-hour
sessions and 30 GPU-hours a week, so it fits — but if you would rather see an
answer sooner, cut `TARGETS` down to one or two folds and run the rest later.

The run is **resumable**: it skips any run already present in the results
file, so a second session picks up where the first stopped (see the last cell
for how to carry results forward).

In [ ]:
import pathlib

# Edit these to control scope.
TARGETS = ["cora", "photo", "ppi", "elliptic"]   # e.g. ["cora", "photo"] for a shorter run
SEEDS = [0, 1, 2]
FRACTIONS = [0.01, 0.05, 0.10, 0.50, 1.00]

# Results land here. /kaggle/working is saved as notebook output and is the
# ONLY directory that survives the session ending.
OUT = pathlib.Path("/kaggle/working/runs_mae.jsonl")

# If you attached a previous run's output as a dataset, seed from it so this
# session resumes instead of starting over.
PREVIOUS = pathlib.Path("/kaggle/input")
if PREVIOUS.exists() and not OUT.exists():
    found = list(PREVIOUS.glob("**/runs_mae.jsonl"))
    if found:
        import shutil
        shutil.copy(found[0], OUT)
        print(f"resuming from {found[0]}")

from src import arms, config
from src.config import RunConfig

planned = arms.matrix_configs(targets=TARGETS, seeds=SEEDS, fractions=FRACTIONS,
                              base=RunConfig(pretrain_objective="mae"))
already = config.completed_run_ids(OUT) if OUT.exists() else set()
todo = [c for c in planned if c.run_id not in already]

print(f"planned  : {len(planned)} runs")
print(f"already  : {len(planned) - len(todo)}")
print(f"to run   : {len(todo)}")
print(f"\nrough estimate: {len(todo) * 70 / 3600:.1f} h  (70s/run average)")

## 3. Run it

Output streams below. Each line is one completed run: arm, label fraction,
score and wall time. `[no plateau]` marks a run that hit its epoch cap while
still improving — its score is a lower bound, not a measurement.

In [ ]:
import subprocess

cmd = [
    sys.executable, "-u", "scripts/run_all.py",
    "--objective", "mae",
    "--out", str(OUT),
    "--targets", *TARGETS,
    "--seeds", *[str(s) for s in SEEDS],
    "--fractions", *[str(f) for f in FRACTIONS],
]
print(" ".join(cmd), "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\nexit code: {process.returncode}")

## 4. Read the answer

The comparison that matters: does the MAE-pretrained encoder beat a
random-init encoder, and does three-source pretraining beat the best single
source? Same verdict rule as before — a gap smaller than its own spread across
seeds is not evidence.

In [ ]:
import pandas as pd

from src import results

pd.set_option("display.width", 200)
df = results.load_frame(OUT)
print(f"{len(df)} runs loaded from this session\n")

for target in [t for t in TARGETS if t in set(df["target"])]:
    sub = df[df["target"] == target]
    summary = results.summarise(sub)
    fractions = sorted(summary["fraction"].unique())
    print(f"=== {target}  ({summary['metric'].iloc[0]}) ===")
    print("  " + " " * 24 + "".join(f"{f:>13.0%}" for f in fractions))
    for arm in results.ARM_ORDER:
        row = summary[summary["arm"] == arm].sort_values("fraction")
        if row.empty:
            continue
        cells_ = "".join(f"{m:>7.3f}±{s:<5.3f}"
                         for m, s in zip(row["mean"], row["std"]))
        print(f"  {results.ARM_LABELS[arm]:<24}{cells_}")
    print()

In [ ]:
gains = results.transfer_gain(df)

print("=" * 66)
print("MAE objective — does cross-domain pretraining transfer?")
print("=" * 66)
for col, label in [("gain_vs_random", "arm A vs random-init"),
                   ("gain_vs_expert", "arm A vs BEST single expert")]:
    if f"{col}_mean" not in gains:
        continue
    v = results.verdict(gains, col)
    counts = v["verdict"].value_counts().to_dict()
    print(f"\n{label}")
    print(f"   cells: {len(v)}")
    for k in ["positive", "indistinguishable", "negative"]:
        print(f"   {k:<20} {counts.get(k, 0)}")
    print(f"   mean gain: {v['mean'].mean():+.4f}")

print()
print("Compare against the DGI study (results/runs.jsonl, local RTX 2050):")
print("   arm A vs random-init        2 positive, 14 indistinguishable, 4 negative")
print("   arm A vs BEST single expert 2 positive,  9 indistinguishable, 9 negative")
print()
print("If MAE lands in the same place, the negative result is about")
print("cross-domain transfer rather than about DGI. If it does not, the")
print("choice of pretext task is what decides transfer -- which is a more")
print("interesting finding than the one we started with.")
print("=" * 66)

## 5. Save the results

**Do this before the session ends or the run is lost.** `/kaggle/working` is
the only directory that persists, and only as saved notebook output.

- **Save Version → Save & Run All** keeps everything automatically, or
- download `runs_mae.jsonl` from the *Output* tab on the right

To continue in a later session: add this notebook's output as an input dataset
to the new one. The cell in section 2 finds `runs_mae.jsonl` under
`/kaggle/input` and resumes from it, skipping everything already finished.

Then, locally:

```bash
# merge the Kaggle results into the repo, keeping the DGI study separate
cp ~/Downloads/runs_mae.jsonl results/runs_mae.jsonl
```

The two files stay separate on purpose — different machines, different torch
versions. Same-objective comparisons happen within a file.

In [ ]:
import shutil

if OUT.exists():
    size_kb = OUT.stat().st_size / 1024
    n = len(config.load_runs(OUT))
    print(f"{OUT}")
    print(f"  {n} runs, {size_kb:.0f} KB")
    print("\nDownload it from the Output tab, or use Save & Run All.")

    # A copy inside the repo folder is convenient but NOT persistent --
    # /kaggle/working/OmniGraph is wiped with the session unless committed.
    shutil.copy(OUT, "/kaggle/working/runs_mae_backup.jsonl")
    print("backup written to /kaggle/working/runs_mae_backup.jsonl")
else:
    print("no results file — did the run cell fail?")